# 🔥 YOLOvX Fine-Tuning Server
> Auto-generated worker for remote GPU training sessions.

**Runtime Setup:** Runtime → Change runtime type → **T4 GPU** → Save, then Run All.

**Flow:** Cell 1 (setup, once) → Cell 2 (start server, trains when dispatched from the website) → Cell 3 (run only when happy with results, uploads the new weights).

In [ ]:
# ⚡ CELL 1 — Install & Download Setup (Run Once)
!pip install -q fastapi uvicorn python-multipart requests ultralytics pydantic

import os

# Download cloudflared binary if missing
if not os.path.exists('cloudflared-linux-amd64'):
    os.system('wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
    os.system('chmod +x cloudflared-linux-amd64')
    print('✅ Cloudflared binary ready.')

# Download finetune_worker.py directly from root of repository
!wget -q -O finetune_worker.py https://raw.githubusercontent.com/tushar0067/notebooks/main/finetune_worker.py
print('✅ Fine-tuning worker script downloaded successfully.')

## 🚀 Cell 2 — Start Server
Run each session. Copy the generated `trycloudflare.com` URL into your Web App's Training Page.

This cell **finishes on its own in a few seconds** — the server and tunnel keep running in the background on daemon threads. That's intentional: it keeps the notebook free so Cell 3 can actually run later instead of queuing forever behind a blocked cell.

Watch this cell's output area — training progress logs will keep appearing here even after it shows as "finished," since the background thread writes to the same output.

In [ ]:
# 🚀 CELL 2 — Start Training Server (Run Each Session)
import threading, uvicorn, subprocess, time, re, sys
sys.path.insert(0, '.')

TUNNEL_URL = None

def start_tunnel():
    global TUNNEL_URL
    p = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:8000'],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(3)
    for line in iter(p.stderr.readline, b''):
        line = line.decode('utf-8')
        if '.trycloudflare.com' in line:
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if m:
                TUNNEL_URL = m.group(0)
                print(f'\n\n🔥 COPY THIS URL INTO YOUR TRAINING PAGE:\n\n  {TUNNEL_URL}\n\n')
                break

def run_server():
    import finetune_worker
    uvicorn.run(finetune_worker.app, host='0.0.0.0', port=8000, log_level='warning')

print('🚀 Starting YOLO Fine-Tuning Server...')
threading.Thread(target=start_tunnel, daemon=True).start()
threading.Thread(target=run_server, daemon=True).start()

# NOTE: no "while True: sleep" here on purpose. The server + tunnel run on
# daemon threads and keep running in the background even after this cell
# finishes — that's what frees up the kernel so Cell 3 (or a re-run of this
# cell with new settings) can actually execute instead of queuing forever.
print('✅ Server + tunnel launched in the background. This cell can now finish —')
print('   the server keeps running. Watch this output for training progress logs.')
time.sleep(5)  # just enough to let the tunnel URL print above before the cell returns

## 📤 Cell 3 — Review & Upload
**Run this ONLY after Cell 2's output shows "Training complete. Results saved for review."**

It prints the final metrics and settings from the last run, asks you to confirm, and only then uploads the new weights — nothing is overwritten automatically. If you're not happy with the results, just re-run Cell 2 with different settings instead; the old model stays untouched.

In [ ]:
# 📤 CELL 3 — Review & Upload (run this ONLY if you're happy with the results)
import json
import requests

SUPABASE_URL = "https://base.wiserly.org"
LAST_RUN_PATH = "/content/last_run.json"

with open(LAST_RUN_PATH) as f:
    run = json.load(f)

print("📊 Last training run:")
print(f"  Model:   {run['model_id']}")
print(f"  Epochs:  {run['epochs']}")
print(f"  Metrics: {run['metrics']}")
print(f"  Settings used: {run['train_kwargs']}")
print()

confirm = input("Upload these weights and overwrite the current model? (yes/no): ").strip().lower()

if confirm != "yes":
    print("❌ Upload cancelled. Nothing was changed. Re-run Cell 2 to try different settings.")
else:
    with open(run["weights_path"], "rb") as f:
        weights_bytes = f.read()

    res = requests.post(
        f"{SUPABASE_URL}/functions/v1/complete-finetune-session",
        headers={"Authorization": f"Bearer {run['session_token']}"},
        files={"file": ("best.pt", weights_bytes, "application/octet-stream")},
        data={"model_id": run["model_id"]},
    )

    if res.status_code == 200:
        print("✅ Uploaded successfully. Your model has been updated.")
    else:
        print(f"❌ Upload failed: {res.text}")